In [ ]:
!pip install wikipedia

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=ed96a7ab703ff1564750cac13c5ae5f74327fced0ef23efd5e9b120fa0bbfb78
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


In [ ]:
import wikipedia

article = wikipedia.page("Artificial intelligence")
document_text = article.content

print(len(document_text))
print(document_text[:500])

85357
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.
High-p


In [ ]:
def chunk_text(text, chunk_size=1000):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunk = text[i:i + chunk_size]
        chunks.append(chunk)
    return chunks

chunks = chunk_text(document_text)

print("Number of chunks:", len(chunks))
print("First chunk:")
print(chunks[0])

Number of chunks: 86
First chunk:
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.
High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.
The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robo

In [ ]:
!pip install sentence-transformers


In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
chunk_embeddings = embedder.encode(chunks)

print(chunk_embeddings.shape)

(86, 384)


In [ ]:
question = "What is machine learning?"
question_embedding = embedder.encode([question])

from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(question_embedding, chunk_embeddings)[0]

best_chunk_index = similarities.argmax()

print("Best matching chunk:")
print(chunks[best_chunk_index])

Best matching chunk:
 many kinds of classifiers in use. The decision tree is the simplest and most widely used symbolic machine learning algorithm. K-nearest neighbor algorithm was the most widely used analogical AI until the mid-1990s, and Kernel methods such as the support vector machine (SVM) displaced k-nearest neighbor in the 1990s.
The naive Bayes classifier is reportedly the "most widely used learner" at Google, due in part to its scalability.
Neural networks are also used as classifiers.


=== Artificial neural networks ===

An artificial neural network is based on a collection of nodes also known as artificial neurons, which loosely model the neurons in a biological brain. It is trained to recognise patterns; once trained, it can recognise those patterns in fresh data. There is an input, at least one hidden layer of nodes and an output. Each node applies a function and once the weight crosses its specified threshold, the data is transmitted to the next layer. A network is typi

In [ ]:
def retrieve_relevant_chunk(question, top_n=2):
    question_embedding = embedder.encode([question])
    similarities = cosine_similarity(question_embedding, chunk_embeddings)[0]

    top_indices = similarities.argsort()[-top_n:][::-1]

    relevant_chunks = [chunks[i] for i in top_indices]
    return relevant_chunks

In [ ]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.3 MB/s eta 0:00:00


In [ ]:
from groq import Groq

client = Groq(api_key="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")
def rag_chat(question):
    relevant_chunks = retrieve_relevant_chunk(question)
    context = "\n\n".join(relevant_chunks)

    system_prompt = f"""You are a helpful assistant. Answer the user's question using ONLY the context below.
      If the answer isn't in the context, say "I don't know based on the provided document."

      Context:
      {context}"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ]
    )

    print(response.choices[0].message.content)

In [ ]:
rag_chat("What is machine learning?")

Machine learning is the study of programs that can improve their performance on a given task automatically.


In [ ]:
rag_chat("What did Albert Einstein have for breakfast?")

In [ ]:
rag_chat("Is AI dangerous? Give me your personal opinion, not what the document says.")

In [ ]:
print(retrieve_relevant_chunk("What are the ethical concerns?"))

In [ ]:
rag_chat("What does this article say about AI regulation in the European Union in 2026?")

In [ ]:
rag_chat("The document says AI will definitely cause mass unemployment by 2030, right?")

In [ ]:
eval_set = [
    {"question": "In what year was artificial intelligence officially founded as an academic discipline?", "expected": "1956"},
    {"question": "Who originally coined the term \"artificial intelligence\"?", "expected": "John McCarthy"},
    {"question": "What specific term is used to describe historical periods where AI research experienced a loss of funding and a wave of institutional disappointment?", "expected": "AI winters"},
    {"question": "Name three high-profile applications of AI listed in the article's introduction.", "expected": "Advanced web search engines, chatbots, virtual assistants, autonomous vehicles, or strategy game engines (like chess and Go)"},
    {"question": "What is the term for a formal representation of knowledge as a set of concepts within a domain and the relationships between them?", "expected": "An ontology"},
    {"question": "Besides engineering, mathematics, and computer science, what are some of the other interdisciplinary fields that AI draws upon?", "expected": "Psychology, linguistics, philosophy, and neuroscience"},
    {"question": "Summarize the traditional goals of AI research as outlined in the article.", "expected": "The core traditional goals include learning, reasoning, knowledge representation, planning, natural language processing, perception, and support for robotics."},
    {"question": "How does an AI agent calculate and use \"expected utility\" to make choices?", "expected": "For each possible action, the agent calculates the utility of all potential outcomes, weights those outcomes by the mathematical probability that they will occur, and then selects the specific action that maximizes this overall value."},
    {"question": "What is the difference between \"machine perception\" and \"computer vision\" based on the text?", "expected": "Machine perception is the broad, overarching capability to use input from any type of sensor (including microphones, lidar, sonar, and radar) to deduce aspects of the physical world. Computer vision is a specific subfield under that umbrella dedicated strictly to analyzing visual input."},
    {"question": "What components make up a Markov decision process according to the article's framework?", "expected": "A Markov decision process relies on a transition model (which describes the probability that a particular action will change the state in a specific way) and a reward function (which supplies the utility of each state and the cost of each action)."},
    {"question": "How does the article explain the real-world utility of formal knowledge representations?", "expected": "They allow AI systems to answer questions intelligently and deduce real-world facts. They are actively applied in clinical decision support, scene interpretation, content-based indexing/retrieval, and automated knowledge discovery in massive databases."},
    {"question": "How does the article characterize the rapid evolution of generative AI since the 2020s?", "expected": "It notes that generative AI has become widely accessible to the public, allowing users to generate high-quality images, audio, and videos purely from basic text prompts."},
    {"question": "What exact percentage of global carbon emissions was directly caused by AI data centers in the year 2025?", "expected": "I don't know. The article notes that power needs and environmental impacts are recognized risks of AI, but it does not provide any specific statistical breakdowns or concrete percentage data for global emissions in 2025."},
    {"question": "How many lines of code did Allen Newell and Herbert A. Simon write to build the initial version of the Logic Theorist program presented at the Dartmouth Workshop?", "expected": "I don't know. The article mentions that the Logic Theorist was presented at the 1956 workshop as the first AI program, but it contains absolutely no information regarding its source code length, language implementation, or code lines."},
    {"question": "Which specific textbook is currently ranked as the absolute most widely used resource for teaching AI in universities worldwide?", "expected": "I don't know. While the text discusses AI as an established academic and scientific research discipline, it does not track, reference, or rank textbook sales, authors, or university syllabi."},
    {"question": "What proprietary algorithms did tech companies use to transition their web search engines from keyword indexing to generative semantic matching?", "expected": "I don't know. The article states that advanced web search engines are a high-profile application of AI, but it does not disclose or detail the internal corporate codebases, proprietary engineering designs, or specific mathematical transitions used by commercial engines."},
    {"question": "What was the exact financial budget surplus or deficit of the 1956 Dartmouth Workshop after attendee registration fees were processed?", "expected": "I don't know. The article outlines the historical significance of the Dartmouth Workshop as the birthplace of AI, but it completely omits any logistical financial records, costs, or registration details of the event."},
    {"question": "Exactly how many parameters are configured within the primary neural network models that handle autonomous vehicle navigation today?", "expected": "I don't know. The text explicitly mentions autonomous vehicles as an application of machine perception and AI, but it never details the technical specifications, architecture layers, or parameter counts of actual modern driving models."}
]

In [ ]:
print(f"Total eval questions: {len(eval_set)}")
print(eval_set[0])

Total eval questions: 18
{'question': 'In what year was artificial intelligence officially founded as an academic discipline?', 'expected': '1956'}


In [ ]:
def get_rag_answer(question, top_n=2):
    relevant_chunks = retrieve_relevant_chunk(question, top_n=top_n)
    context = "\n\n".join(relevant_chunks)

    system_prompt = f"""You are a helpful assistant. Answer the user's question using ONLY the context below.
If the answer isn't in the context, say "I don't know based on the provided document."

Context:
{context}"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ]
    )

    return response.choices[0].message.content

In [ ]:
results = []

for item in eval_set:
    question = item["question"]
    expected = item["expected"]
    actual = get_rag_answer(question)

    results.append({
        "question": question,
        "expected": expected,
        "actual": actual
    })

print(f"Ran {len(results)} questions")

Ran 18 questions


In [ ]:
for r in results:
    print("Q:", r["question"])
    print("Expected:", r["expected"])
    print("Actual:", r["actual"])
    print("---")

Q: In what year was artificial intelligence officially founded as an academic discipline?
Expected: 1956
Actual: The field of AI research was founded as an academic discipline at a workshop at Dartmouth College in 1956.
---
Q: Who originally coined the term "artificial intelligence"?
Expected: John McCarthy
Actual: I don't know based on the provided document.
---
Q: What specific term is used to describe historical periods where AI research experienced a loss of funding and a wave of institutional disappointment?
Expected: AI winters
Actual: These historical periods are referred to as "winters" for AI. According to the context, there were at least two winters, one which began after the collapse of the Lisp Machine market in 1987, and a first wave prior to that mentioned indirectly.
---
Q: Name three high-profile applications of AI listed in the article's introduction.
Expected: Advanced web search engines, chatbots, virtual assistants, autonomous vehicles, or strategy game engines (lik

In [ ]:
def judge_answer(question,expected, actual):

  judge_prompt=f"""You are grading whether an AI's answer is correct.

Question: {question}
Expected answer: {expected}
AI's actual answer: {actual}

Does the AI's actual answer correctly capture the expected answer, even if worded differently?
Respond with ONLY "PASS" or "FAIL", followed by a one-sentence reason.
Format exactly like this:
PASS: reason here
or
FAIL: reason here"""

  response = client.chat.completions.create(
      model="llama-3.1-8b-instant",
      messages=[
          {"role": "user", "content": judge_prompt},
      ]
  )
  return response.choices[0].message.content

In [ ]:


for r in results:
  verdict= judge_answer(r["question"],r["expected"],r["actual"])
  r["verdict"]= verdict
  print(verdict)
  print("---")

PASS: The AI's actual answer accurately conveys the correct year and provides additional context which does not detract from the core answer.
---
FAIL: The AI's answer does not acknowledge the correct information that John McCarthy originally coined the term "artificial intelligence".
---
PASS: Although the AI's answer is phrased somewhat differently, it accurately captures the definition and specific term to describe the historical periods in question.
---
PASS: The AI's actual answer correctly lists three high-profile applications of AI, matching the expected answer's content, although phrased slightly differently.
---
PASS: The AI's answer accurately describes an ontology, although it is not phrased identically to the expected answer.
---
FAIL: The AI's answer does not capture the expected answer as it explicitly states a lack of knowledge, rather than providing the relevant interdisciplinary fields.
---
FAIL: The AI's actual answer excludes key traditional goals of AI research such

In [ ]:
passes= sum(1 for r in results if r["verdict"].startswith("PASS"))
total= len(results)

print(f"Score: {passes}/{total} ({passes/total*100:.1f}%)")

Score: 14/18 (77.8%)


In [ ]:
baseline_score = passes / total
print(f"Baseline score: {baseline_score*100:.1f}%")

Baseline score: 77.8%


In [ ]:
def get_rag_answer_v2(question, top_n=4):
    relevant_chunks = retrieve_relevant_chunk(question, top_n=top_n)
    context = "\n\n".join(relevant_chunks)

    system_prompt = f"""You are a helpful assistant. Answer the user's question using ONLY the context below.
If the answer isn't in the context, say "I don't know based on the provided document."

Context:
{context}"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ]
    )

    return response.choices[0].message.content, response.usage.total_tokens

In [ ]:
results_v1 = []
total_tokens_used = 0

for item in eval_set:
    question = item["question"]
    expected = item["expected"]
    actual , tokens = get_rag_answer_v2(question, top_n=2)

    verdict = judge_answer(question, expected, actual)

    total_tokens_used += tokens

    results_v1.append({
        "question": question,
        "expected": expected,
        "actual": actual,
        "verdict": verdict,
        "tokens": tokens
    })

print(f"Total tokens used for {len(eval_set)} questions: {total_tokens_used}")
print(f"Average tokens per question: {total_tokens_used / len(eval_set):.0f}")

passes_v1 = sum(1 for r in results_v1 if r["verdict"].startswith("PASS"))
score_v1 = passes_v1 / len(results_v1)

print(f"Baseline (top_n=2): {baseline_score*100:.1f}%")

Total tokens used for 18 questions: 9135
Average tokens per question: 508
Baseline (top_n=2): 77.8%


In [ ]:
results_v2 = []
total_tokens_used = 0

for item in eval_set:
    question = item["question"]
    expected = item["expected"]
    actual , tokens = get_rag_answer_v2(question)

    verdict = judge_answer(question, expected, actual)

    total_tokens_used += tokens

    results_v2.append({
        "question": question,
        "expected": expected,
        "actual": actual,
        "verdict": verdict,
        "tokens": tokens
    })

print(f"Total tokens used for {len(eval_set)} questions: {total_tokens_used}")
print(f"Average tokens per question: {total_tokens_used / len(eval_set):.0f}")

passes_v2 = sum(1 for r in results_v2 if r["verdict"].startswith("PASS"))
score_v2 = passes_v2 / len(results_v2)

print(f"Baseline (top_n=2): {baseline_score*100:.1f}%")
print(f"Version 2 (top_n=4): {score_v2*100:.1f}%")

Total tokens used for 18 questions: 16040
Average tokens per question: 891
Baseline (top_n=2): 77.8%
Version 2 (top_n=4): 88.9%


In [ ]:
for i in range(len(results)):
    if results[i]["verdict"].startswith("PASS") != results_v2[i]["verdict"].startswith("PASS"):
        print("Question:", results[i]["question"])
        print("v1 verdict:", results[i]["verdict"])
        print("v2 verdict:", results_v2[i]["verdict"])
        print("---")

In [ ]:
def get_rag_answer_v3(question, top_n=2):
    relevant_chunks = retrieve_relevant_chunk(question, top_n=top_n)
    context = "\n\n".join(relevant_chunks)

    system_prompt = f"""You are a precise research assistant. Answer STRICTLY and ONLY using the context below.
Do not use any outside knowledge, even if you know the answer.
If the context does not fully answer the question, respond exactly: "I don't know based on the provided document."
Be concise — 1-3 sentences maximum.

Context:
{context}"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ]
    )

    return response.choices[0].message.content

In [ ]:
results_v3 = []

for item in eval_set:
    question = item["question"]
    expected = item["expected"]
    actual = get_rag_answer_v3(question)

    verdict = judge_answer(question, expected, actual)

    results_v3.append({
        "question": question,
        "expected": expected,
        "actual": actual,
        "verdict": verdict
    })

passes_v3 = sum(1 for r in results_v3 if r["verdict"].startswith("PASS"))
score_v3 = passes_v3 / len(results_v3)

print(f"Baseline (top_n=2): {baseline_score*100:.1f}%")
print(f"Version 2 (top_n=4): {score_v2*100:.1f}%")
print(f"Version 3 (top_n=2 with changed prompt): {score_v3*100:.1f}%")

In [ ]:
chunks_v4 = chunk_text(document_text, chunk_size=2000)
chunk_embeddings_v4 = embedder.encode(chunks_v4)
print(chunk_embeddings_v4.shape)

def retrieve_relevant_chunk(question, top_n=2):
    question_embedding = embedder.encode([question])
    similarities = cosine_similarity(question_embedding, chunk_embeddings_v4)[0]

    top_indices = similarities.argsort()[-top_n:][::-1]

    relevant_chunks = [chunks[i] for i in top_indices]
    return relevant_chunks

def get_rag_answer_v4(question, top_n=2):
    relevant_chunks = retrieve_relevant_chunk(question, top_n=top_n)
    context = "\n\n".join(relevant_chunks)

    system_prompt = f"""You are a helpful assistant. Answer the user's question using ONLY the context below.
If the answer isn't in the context, say "I don't know based on the provided document."

Context:
{context}"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ]
    )

    return response.choices[0].message.content



In [ ]:
results_v4 = []

for item in eval_set:
    question = item["question"]
    expected = item["expected"]
    actual = get_rag_answer_v4(question)

    verdict = judge_answer(question, expected, actual)

    results_v4.append({
        "question": question,
        "expected": expected,
        "actual": actual,
        "verdict": verdict
    })

passes_v4 = sum(1 for r in results_v4 if r["verdict"].startswith("PASS"))
score_v4 = passes_v4 / len(results_v4)

print(f"Baseline (top_n=2): {baseline_score*100:.1f}%")
print(f"Version 2 (top_n=4): {score_v2*100:.1f}%")
print(f"Version 3 (top_n=2 with changed prompt): {score_v3*100:.1f}%")
print(f"Version 4 (changed chunks to 2000): {score_v4*100:.1f}%")

In [ ]:
print(f"{'Version':<25} {'Score':<10}")
print(f"{'Baseline (top_n=2)':<25} {baseline_score*100:.1f}%")
print(f"{'top_n=4':<25} {score_v2*100:.1f}%")
print(f"{'Stricter prompt':<25} {score_v3*100:.1f}%")
print(f"{'chunk_size=2000':<25} {score_v4*100:.1f}%")